In [1]:

%load_ext sql
%reload_ext sql
%config SqlMagic.displaylimit = 10
%sql sqlite:///../data/thumbnail_model.db
%config SqlMagic.feedback = 1

%load_ext autoreload
%autoreload 2

# add to path
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

import pandas as pd

pd.set_option('display.width', 1024)
pd.set_option('display.max_colwidth', 256)

import json

Connecting to 'sqlite:///../data/thumbnail_model.db'

In [7]:
%%sql
DROP VIEW IF EXISTS v_label_names;
CREATE TEMP VIEW v_label_names AS
    SELECT 
        REPLACE(video_id, '.jpg', '') AS video_id, 
        GROUP_CONCAT(json_extract(value, '$.Name')) AS labels
    FROM 
        rekognition_responses,
        json_each(json_extract(label_response, '$.Labels'))
    GROUP BY 
        video_id;

Running query in 'sqlite:///../data/thumbnail_model.db'

++
||
++
++

In [11]:
label_names = %sql SELECT * FROM v_label_names;
label_names_df = label_names.DataFrame().set_index('video_id')
label_names_df['labels'] = label_names_df['labels'].apply(lambda x: x.split(','))
label_names_df

Running query in 'sqlite:///../data/thumbnail_model.db'

,labels
video_id,
--G_CA9W3_4,"[Lighting, Purple, Adult, Male, Man, Person, People, Light, Advertisement, Poster, Photography, Clothing, Footwear, Shoe, Sneaker, Boot, Book, Publication, Comics, Club, Flare]"
--LF1-D0V2U,"[Adult, Bride, Female, Person, Wedding, Woman, Book, Publication, Plant, Vegetation, Outdoors, Nature, Photography, Angel, Samurai, Weapon, Clothing, Costume]"
--ObpVwtfQE,"[Indoors, Interior Design, Outdoors, Person, Nature, Gun, Weapon, Dungeon, Shooting, Sewer, Night, Storm, Tornado, Smoke]"
--eCdoJdTOg,"[Hunting, Adult, Female, Person, Woman, Animal, Food, Invertebrate, Lobster, Sea Life, Seafood, Samurai, Sniper, Photography]"
-0BK0235PVA,"[Adult, Female, Person, Woman, Male, Man, Treasure, Indoors, Alien, Aircraft, Transportation, Vehicle, Sword, Weapon]"
...,...
zyfX1CLuVDU,"[Aquatic, Water, Animal, Aquarium, Fish, Sea Life, Adult, Female, Person, Woman, Outdoors, Nature, Sea, Blackboard, Reef, Adventure, Leisure Activities, Scuba Diving, Sport, Coral Reef, Pollution, Clothing, Coat, Angler, Fishing]"
zyoj5fFeCzc,"[Advertisement, Blade, Dagger, Knife, Weapon, Poster, Text, Sink, Sink Faucet, Device]"
zytZ84tYd4Q,"[Book, Publication, Plant, Vegetation, Outdoors, Nature, Advertisement, Water, Animal, Mammal, Poster, Sky, Bear, Wildlife, Jungle, Art, Transportation, Vehicle, Land, Graphics, Landscape, Comics]"


In [12]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit(label_names_df['labels'])
label_matrix = mlb.transform(label_names_df['labels'])
label_matrix_df = pd.DataFrame(label_matrix, columns=mlb.classes_, index=label_names_df.index)
label_matrix_df

,Absinthe,Abyssinian,Accessories,Acorn,Acrobatic,Action Figure,Adapter,Adult,Adventure,Advertisement,...,Wristwatch,Writing,X-Ray,Yacht,Yak,Yard,Yawning,Yoga,Zebra,Zoo
video_id,,,,,,,,,,,,,,,,,,,,,
--G_CA9W3_4,0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
--LF1-D0V2U,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
--ObpVwtfQE,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
--eCdoJdTOg,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
-0BK0235PVA,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zyfX1CLuVDU,0,0,0,0,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
zyoj5fFeCzc,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
zytZ84tYd4Q,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [13]:
view_count = %sql SELECT video_id, view_count FROM statistics;
view_count_df = view_count.DataFrame().set_index('video_id')
view_count_df

Running query in 'sqlite:///../data/thumbnail_model.db'

,view_count
video_id,
XjrDeIujT-I,16998
ZdX4E2UO_QY,99618
n9rN5OLOx5Y,3708
-VitUNlkK8M,46863
FpQhTbDw2v4,7004
...,...
PVlUX_lbDAQ,56011
TSzbGP5FC7A,24432
HT02SgLZ6R8,260582


In [22]:
indices = list(set(label_matrix_df.index) & set(view_count_df.index))
X_df = label_matrix_df.loc[indices]
y_df = view_count_df.loc[indices]

y_df

,view_count
video_id,
ML5K2pHS5tA,774
Q43zf6D6REk,4135
w8Hwnkumeo0,1002756
1qtEmxRYlbA,313266
fwDlfEY2aBc,23
...,...
7SenaUqLw_4,1403265
O0VE3aXKV30,306
jTA96eRsuFg,632441
